In [1]:
import json
from portfolio import Portfolio
from optimizer import PortfolioOptimizer
from simulations import MonteCarloSimulation
from utils import (
    get_simulation_insights,
    display_optimal_weights
)

#Snowpark lib
from snowflake.snowpark import Session
import pandas as pd
from fosforml.model_manager.snowflakesession import get_session
my_session = get_session()

config = {
    "TICKERS" : [], "START_DATE" : '2020-01-01', "END_DATE" : '2023-01-01', "INITIAL_INVESTMENT" : 100000,
    "NUM_SIMULATIONS" : 500, "TIME_HORIZON" : 252, "RISK_FREE_RATE" : 0.02, "WEIGHTS" : None, "OPTIMIZE" : True, "BALANCED" : False
}

In [2]:
table_name = 'STOCKS_DATA'
sf_df = my_session.sql("select * from {}".format(table_name))
df = sf_df.to_pandas()

In [3]:
df["STOCK"].unique()

array(['MTFS', 'ALPA', 'OOGGL', 'NTIC', 'CCSO', 'MZNA', 'LTSA', 'HD',
       'DMC', 'NEK', 'JJN', 'EPF', 'KRM', 'ABT', 'GMNA', 'MPJ', 'ABC',
       'FWC', 'GS', 'SM', 'AB', 'TCA', 'EG', 'MMM', 'ONH', 'PG', 'OK',
       'PEP', 'TWM', 'OCST'], dtype=object)

In [4]:
tickers = ["MTFS", "ALPA", "TCA", "NTIC", 'MZNA', 'EG', 'HD']
config["TICKERS"] = tickers

In [5]:
def load_data(df, tickers: list):
    stock_data = {}
    min_len = []
    for ticker in tickers:
        data = df[df["STOCK"] == ticker]
        stock_data[ticker] = list(data["ADJ_CLOSE"])
        min_len.append(len(data))
    last_len = min(min_len)
    for ticker in tickers:
        stock_data[ticker] = stock_data[ticker][:last_len]
    
    stock_data = pd.DataFrame(stock_data)
    return stock_data

load_data(df, config["TICKERS"]).head()

,MTFS,ALPA,TCA,NTIC,MZNA,EG,HD
0,23.347319,6.454505,40.027218,13.519428,6.6950,53.996017,20.165382
1,23.354864,6.465664,40.505753,13.512954,6.7345,54.275623,20.313089
2,23.211533,6.362820,40.628815,13.467625,6.6125,53.996017,20.242744
3,22.970140,6.351057,40.792885,13.338132,6.5000,56.791931,20.481892
4,23.128557,6.393279,41.250919,13.487052,6.6760,58.015163,20.383423


In [8]:
def main(config=config):
    
    # Extract configuration parameters
    tickers = config.get('TICKERS', ["MTFS", "ALPA", "OOGGL"])
    start_date = config.get('START_DATE')
    end_date = config.get('END_DATE')
    initial_investment = config.get('INITIAL_INVESTMENT', 1000)
    num_simulations = config.get('NUM_SIMULATIONS', 10000)
    time_horizon = config.get('TIME_HORIZON', 252)
    risk_free_rate = config.get('RISK_FREE_RATE', 0.0)
    custom_weights = config.get('WEIGHTS')
    optimization_config = config.get('optimization', {})
    optimize = optimization_config.get('OPTIMIZE', True)
    balanced = optimization_config.get('BALANCED', False)
    # Load data
    stock_data = load_data(df, tickers)
    
    # Create portfolio
    One = Portfolio(stock_data)
    One.calculate_returns()
    
    # Annualize returns and covariance
    expected_returns = One.returns.mean() * 252
    covariance_matrix = One.returns.cov() * 252

    # return covariance_matrix, expected_returns
    
    # Determine weights
    if optimize:
        optimizer = PortfolioOptimizer(
            expected_returns,
            covariance_matrix,
            risk_free_rate=risk_free_rate
        )
        if balanced:
            optimal_weights = optimizer.minimize_volatility(target_return=expected_returns.mean())
            # print("\nOptimal Balanced Portfolio Weights:")
        else:
            optimal_weights = optimizer.maximize_sharpe_ratio()
            # print("\nOptimal Portfolio Weights to Maximize Sharpe Ratio:")
        display_optimal_weights(stock_data.columns, optimal_weights)
        weights = optimal_weights
    elif custom_weights:
        weights = custom_weights
        # print("\nUsing Custom Weights:")
        display_optimal_weights(stock_data.columns, weights)
    else:
        num_assets = len(expected_returns)
        weights = [1.0 / num_assets] * num_assets
        # print("\nUsing Equal Weights:")
        # display_optimal_weights(stock_data.columns, weights)

    # print(weights)
    # Perform Monte Carlo Simulation
    simulation = MonteCarloSimulation(One.returns, initial_investment, weights)
    all_cumulative_returns, final_portfolio_values = simulation.run_simulation(
        num_simulations, time_horizon
    )
    
    # Analyze Results
    final_insights = get_simulation_insights(final_portfolio_values, initial_investment)
    # print("**************************")
    # for k, v in final_insights.items():
    #     print(k,":", v)
    # print(final_insights)
    return all_cumulative_returns, final_portfolio_values
    
    # Plot results
    # plot_simulation_results(all_cumulative_returns, final_portfolio_values)



a, b = main()

  Ticker  Weight
0   MTFS  0.0554
1   ALPA  0.3333
2    TCA  0.0384
3   NTIC  0.0000
4   MZNA  0.1656
5     EG  0.0000
6     HD  0.4073
